# SME-DataTrust BI — configuración
Ejecuta los notebooks del 01 al 06 en orden y dentro de la misma sesión de Colab. Si tu carpeta del proyecto está en otra ruta, modifica `PROJECT_DIR`.

In [ ]:
from pathlib import Path
import sys
candidates=[Path("/content/BI-PYMES-ML"),Path("/content/SME-DataTrust-BI"),Path.cwd()]
PROJECT_DIR=next((p for p in candidates if (p/"src").exists()),Path.cwd())
if not (PROJECT_DIR/"src").exists():
    raise FileNotFoundError("No se encontró src/. Ajusta PROJECT_DIR a la raíz del repositorio.")
sys.path.insert(0,str(PROJECT_DIR))
for d in ["data/raw","data/processed","data/experimental","results/figures","results/metrics"]:
    (PROJECT_DIR/d).mkdir(parents=True,exist_ok=True)
print("Proyecto:",PROJECT_DIR)

## 4. Isolation Forest
El parámetro `contamination` representa una fracción esperada configurada para el algoritmo; no es una medición conocida de errores reales.

In [ ]:
import pandas as pd, joblib
from src.anomaly_detection import fit_isolation_forest
p=PROJECT_DIR/"data/processed/invoice_features.pkl"
if not p.exists(): raise FileNotFoundError("Ejecuta primero Notebook 03.")
invoice=pd.read_pickle(p)
model,scored,features=fit_isolation_forest(invoice,contamination=.03,random_state=42)
print("Variables:",features,"\nSeñaladas:",int(scored.if_anomaly.sum()))
display(scored.sort_values("if_score",ascending=False).head(15))
scored.to_csv(PROJECT_DIR/"results/metrics/isolation_forest_scores.csv",index=False)
joblib.dump(model,PROJECT_DIR/"results/metrics/isolation_forest_pipeline.joblib")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4)); plt.hist(scored.if_score,bins=50)
plt.title("Puntaje de anomalía — Isolation Forest")
plt.xlabel("Mayor puntaje = más atípico"); plt.tight_layout()
plt.savefig(PROJECT_DIR/"results/figures/isolation_forest_scores.png",dpi=160); plt.show()